In [0]:
import os, re, sys
sys.path.append(os.path.abspath('..'))

from pyspark.sql import functions as F
from src.config import CATALOGO, CHECKPOINTS, FONTES_BRONZE, RAW

dbutils.widgets.text('tabelas', 'todas')
filtro = [t.strip() for t in dbutils.widgets.get('tabelas').split(',')]


def normalizar(nome):
    '''Delta não aceita espaços e certos símbolos em nomes de coluna.'''
    return re.sub(r'[^a-z0-9_]+', '_', nome.strip().lower()).strip('_')


def tem_arquivos(pasta):
    for _, _, arquivos in os.walk(pasta):
        if any(not a.startswith(('_', '.')) for a in arquivos):
            return True
    return False


def ingerir_bronze(tabela, pasta, formato, opcoes):
    base_ckpt = f'{CHECKPOINTS}/bronze/{tabela}'
    leitor = (spark.readStream.format('cloudFiles')
              .option('cloudFiles.format', formato)
              .option('cloudFiles.schemaLocation', f'{base_ckpt}/_schema'))
    for chave, valor in opcoes.items():
        leitor = leitor.option(chave, valor)

    df = (leitor.load(f'{RAW}/{pasta}/')
          .withColumn('_arquivo_origem', F.col('_metadata.file_path'))
          .withColumn('_data_ingestao', F.current_timestamp()))
    df = df.toDF(*[c if c.startswith('_') else normalizar(c) for c in df.columns])

    (df.writeStream
       .option('checkpointLocation', f'{base_ckpt}/_checkpoint')
       .option('mergeSchema', 'true')
       .trigger(availableNow=True)
       .toTable(f'{CATALOGO}.bronze.{tabela}')
       .awaitTermination())


erros = []
for tabela, pasta, formato, opcoes in FONTES_BRONZE:
    if 'todas' not in filtro and tabela not in filtro:
        continue
    if not tem_arquivos(f'{RAW}/{pasta}'):
        print(f'[pulado] {tabela}: ainda sem arquivos na Raw')
        continue
    try:
        ingerir_bronze(tabela, pasta, formato, opcoes)
        print(f'[ok] {tabela}')
    except Exception as e:
        erros.append(f'{tabela}: {type(e).__name__}: {str(e)[:300]}')
        print(f'[erro] {tabela}')

if erros:
    raise RuntimeError('Falhas na Bronze -> ' + ' | '.join(erros))